# Books & Interactions Processing Pipeline: Fantasy & Paranormal

This notebook implements a self-contained data pipeline for the Goodreads Fantasy category. Every step is grounded in findings from [EDA_Fantasy.ipynb](EDA_Fantasy.ipynb) with exact statistics cited for justification.

## Section 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gzip
import json
import re
import html
import sys

# Add src to path if not already there
sys.path.append('..')
from src.utils.io import read_jsonl_sample
from src.utils.cleaning import empty_strings_to_na, normalize_review_text

CATEGORY = 'fantasy'
RAW_BOOKS_PATH = Path('../data/raw/goodreads_books_fantasy_paranormal.json.gz')
RAW_INTERACTIONS_PATH = Path('../data/raw/goodreads_interactions_fantasy_paranormal.json.gz')
OUTPUT_DIR = Path(f'../data/processed/{CATEGORY}/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Section 1 — Inline Helper Functions

In [2]:
def parse_goodreads_dates(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    return df

def clean_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_authors(authors_list):
    if not isinstance(authors_list, list) or len(authors_list) == 0:
        return pd.Series({'primary_author_id': np.nan, 'author_count': 0})
    primary = [a['author_id'] for a in authors_list if a.get('role', '') == '']
    return pd.Series({
        'primary_author_id': primary[0] if primary else authors_list[0]['author_id'],
        'author_count': len(authors_list)
    })

def extract_series(series_list):
    if not isinstance(series_list, list):
        return pd.Series({'series_count': 0})
    return pd.Series({'series_count': len(series_list)})

def extract_shelves(shelves_list):
    if not isinstance(shelves_list, list) or len(shelves_list) == 0:
        return pd.Series({'top_shelf': np.nan, 'to_read_count': 0})
    to_read = [int(s['count']) for s in shelves_list if 'to-read' in s['name'].lower()]
    return pd.Series({
        'top_shelf': shelves_list[0]['name'],
        'to_read_count': sum(to_read) if to_read else 0
    })

def encode_engagement(row):
    if row['is_read'] and not pd.isna(row['rating_clean']): return 'rated'
    if not pd.isna(row['review_text_incomplete']) and len(row['review_text_incomplete']) > 10: return 'reviewed'
    if row['is_read']: return 'read_no_rating'
    return 'shelf_only'

def cap_at_p99(series):
    p99 = series.quantile(0.99)
    return series.clip(upper=p99)

def empty_strings_to_na(df):
    return df.replace(r'^\s*$', np.nan, regex=True)

def read_jsonl_sample(path, nrows=10000):
    data = []
    with gzip.open(path, 'rt') as f:
        for i, line in enumerate(f):
            if i >= nrows: break
            data.append(json.loads(line))
    return pd.DataFrame(data)

## Section 2 — Books Pipeline

In [3]:
### 2.1 Load
# Loading the raw books dataset using the sample helper.
books = read_jsonl_sample(RAW_BOOKS_PATH, nrows=10000)
print(f"Loaded {len(books)} books")

Loaded 10000 books


In [4]:
books = empty_strings_to_na(books)

### 2.2 Empty strings → NA
**EDA §3 finding**: all missing values are encoded as empty strings. Specific sparsity: `edition_information` 89.7%, `asin` 72.6%, `kindle_asin` 51.6%, `isbn` 49.9% (Fantasy). No true `NaN` exists in the raw JSON.

### 2.3 Numeric columns
**EDA §3 finding**: numeric fields (`ratings_count`, `text_reviews_count`, `num_pages`, `publication_year`, `publication_month`, `publication_day`) arrive as dtype `str` in the raw JSON.

In [5]:
num_cols = ['ratings_count', 'text_reviews_count', 'num_pages', 'publication_year', 'publication_month', 'publication_day']
for col in num_cols:
    books[col] = pd.to_numeric(books[col], errors='coerce')

### 2.4 `is_ebook` boolean
**EDA §3 finding**: field is a raw string `"true"/"false"`. Converted to boolean for consistent typing and retained for format-based segmentation.

In [6]:
books['is_ebook'] = books['is_ebook'].map({'true': True, 'false': False})

### 2.5 Author extraction + role filter
**EDA §4 finding**: of all author entries in Fantasy, 48,987 roles are empty string (`role=""`) — these are primary authors. Non-empty roles include Editor (598), Pseudonym (36), etc. Using entries without a role filter contaminates `primary_author_id`.

In [7]:
author_features = books['authors'].apply(extract_authors)
books = pd.concat([books, author_features], axis=1)

### 2.6 Series extraction + `is_in_series`
**EDA §6 Fantasy finding**: 72.5% of books are in ≥1 series. Series books have an average rating of 3.989 vs 3.822 for standalones (+0.167), indicating strong survivorship bias.

In [8]:
series_features = books['series'].apply(extract_series)
books = pd.concat([books, series_features], axis=1)
books['is_in_series'] = books['series_count'] > 0

### 2.7 Shelves + `to_read_count`
**EDA §5 finding**: `to_read_count` is a demand/hype signal with r≈0.09 correlation with `average_rating` (weak positive). Fantasy scale: mean=6,202, median=388, max=543,151. It must be exposed separately from genre shelves.

In [9]:
shelf_features = books['popular_shelves'].apply(extract_shelves)
books = pd.concat([books, shelf_features], axis=1)

### 2.8 `publication_year_clean`
**EDA §2/§10 Fantasy finding**: `publication_year` range is 12–29,017; 20 records before 1450 and 8 after 2026 are explicit data entry errors. Clean version clips to [1450, 2026].

In [10]:
books['publication_year_clean'] = np.where(
    books['publication_year'].between(1450, 2026),
    books['publication_year'],
    pd.NA
)

### 2.9 `has_isbn`
**EDA §3 finding**: 49.9% (24,974/50,000) are missing ISBN in Fantasy. A presence flag is required for modeling.

In [11]:
books['has_isbn'] = books['isbn'].notna()

### 2.10 Dedup by `book_id`
**EDA §12 finding**: 0 duplicate `book_id` confirmed, but 13,430 duplicate `work_id` (different editions). Dedup on `book_id` is a safety step.

In [12]:
books = books.drop_duplicates('book_id', keep='first')

## Section 3 — Interactions Pipeline

### 3.1 Load
Large file (2.7 GB compressed). Using chunked iterator to avoid OOM.

In [13]:
chunks = []
chunk_iter = pd.read_json(RAW_INTERACTIONS_PATH, lines=True, chunksize=200_000, compression='gzip')

total_processed = 0
limit = 10000

for chunk in chunk_iter:
    chunk = empty_strings_to_na(chunk)
    chunk = parse_goodreads_dates(chunk, ['date_added', 'date_updated', 'read_at', 'started_at'])
    chunk['rating_missing'] = (chunk['rating'] == 0)
    chunk['rating_clean'] = chunk['rating'].where(chunk['rating'].between(1, 5))
    text_col = 'review_text' if 'review_text' in chunk.columns else 'review_text_incomplete'
    chunk['review_text_clean'] = chunk[text_col].apply(normalize_review_text)
    chunk['has_review_text'] = chunk['review_text_clean'].notna()
    chunk['reading_duration_days'] = (chunk['read_at'] - chunk['started_at']).dt.days
    chunk.loc[chunk['reading_duration_days'] < 0, 'reading_duration_days'] = np.nan
    chunk['has_reading_duration'] = chunk['reading_duration_days'].notna()
    
    def get_mode(row):
        if row['is_read'] == False and row['rating_missing'] and not row['has_review_text']:
            return 'shelf_only'
        if pd.notna(row['rating_clean']) and row['has_review_text']:
            return 'reviewed'
        if pd.notna(row['rating_clean']):
            return 'rated'
        return 'read_no_rating'
    
    chunk['engagement_mode'] = chunk.apply(get_mode, axis=1)
    
    # Check if we need to slice the chunk to respect the limit
    if total_processed + len(chunk) > limit:
        chunk = chunk.iloc[:limit - total_processed]
    
    chunks.append(chunk)
    total_processed += len(chunk)
    
    if total_processed >= limit:
        break

interactions = pd.concat(chunks)
print(f"Processed {len(interactions)} interactions")

/tmp/ipykernel_117909/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_117909/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_117909/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_117909/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

Processed 10000 interactions


### 3.8 Dedup by `review_id`
**EDA §12 finding**: 0 duplicate `review_id` confirmed. Strategy: keep row with latest `date_updated` as a precaution.

In [14]:
interactions = interactions.sort_values('date_updated').drop_duplicates('review_id', keep='last')

## Section 4 — User-Level Aggregates
**Justification (EDA §8 Fantasy)**: 2,112 users with ≥1 rating; 53.8% generous (mean ≥ 3.67). Collaborative filtering requires `user_rating_bias` adjustment.

In [15]:
global_mean = interactions['rating_clean'].mean()
user_aggs = interactions[interactions['rating_clean'].notna()].groupby('user_id').agg(
    user_mean_rating=('rating_clean', 'mean'),
    user_rating_std=('rating_clean', 'std'),
    user_rating_count=('rating_clean', 'count')
).reset_index()

user_aggs['user_rating_bias'] = user_aggs['user_mean_rating'] - global_mean
interactions = interactions.merge(user_aggs, on='user_id', how='left')

## Section 5 — Book-Level Aggregates
**Justification (EDA §9)**: 78.5% of books have <1 interaction; 98.3% have <10. The `is_cold_start` flag is essential for routing.

In [16]:
books['book_id'] = books['book_id'].astype(str)
interactions['book_id'] = interactions['book_id'].astype(str)

book_aggs = interactions.groupby('book_id').agg(
    interaction_count=('book_id', 'count'),
    mean_rating=('rating_clean', 'mean')
).reset_index()

book_aggs['is_cold_start'] = book_aggs['interaction_count'] < 10

# Drop before merge to prevent _x/_y suffixes on re-runs
if 'is_cold_start' in books.columns:
    books = books.drop(columns=['is_cold_start', 'interaction_count', 'mean_rating'], errors='ignore')

books = books.merge(book_aggs, on='book_id', how='left')
books['is_cold_start'] = books['is_cold_start'].fillna(True).astype(bool)

if 'user_rating_bias' in interactions.columns:
    interactions = interactions.drop(columns=['user_rating_bias', 'user_mean_rating', 'user_rating_std', 'user_rating_count'], errors='ignore')

interactions = interactions.merge(user_aggs[['user_id', 'user_rating_bias']], on='user_id', how='left')
interactions['user_rating_bias'] = interactions['user_rating_bias'].fillna(0)

## Section 6 — Outlier Treatment
**Justification (EDA §11 Fantasy)**: `ratings_count` has 7,215 IQR outliers (max 534,960). P99-capped versions are added for modeling.

In [17]:
for col in ['ratings_count', 'text_reviews_count', 'num_pages', 'interaction_count']:
    if col in books.columns:
        books[f'{col}_p99'] = cap_at_p99(books[col].fillna(0))

## Section 7 — Output

In [18]:
# Ensure output directory exists (handled in Section 0)
books.to_parquet(OUTPUT_DIR / 'books_curated.parquet', index=False)
interactions.to_parquet(OUTPUT_DIR / 'interactions_curated.parquet', index=False)
print(f"Saved curated files to {OUTPUT_DIR}")

Saved curated files to ../data/processed/fantasy


## Section 8 — Validation

In [19]:
# Original assertions
assert books['book_id'].notna().all()
assert interactions['book_id'].notna().all()
assert interactions['rating_clean'].dropna().between(1, 5).all()
assert books['book_id'].duplicated().sum() == 0
assert interactions['review_id'].dropna().duplicated().sum() == 0

# New (EDA-grounded) assertions
assert books['publication_year_clean'].dropna().between(1450, 2026).all()
valid_modes = {'shelf_only', 'rated', 'reviewed', 'read_no_rating'}
assert interactions['engagement_mode'].isin(valid_modes).all()
assert interactions['reading_duration_days'].dropna().ge(0).all()
assert books['is_cold_start'].dtype == bool or books['is_cold_start'].dtype == 'bool'
assert interactions['user_rating_bias'].notna().sum() > 0

print("All validations passed!")

All validations passed!
